In [1]:
import os
import math
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer,AutoModelForCausalLM

torch.manual_seed(42)
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("运行设备：",device)

C:\Users\Administrator\.conda\envs\rl\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


运行设备： cpu


## 1. 页面因果掩码代码的可运行版本

In [2]:
seq_len=6
'''GPT 等自回归语言模型最核心的机制——因果注意力掩码（Causal Attention Mask）。简单来说，它的作用是确保模型在预测当前位置的 Token 时，只能看到它自己以及它左边的 Token，绝对不能“偷看”到右边的未来 Token。'''
future_mask=torch.triu(torch.ones(seq_len,seq_len,dtype=torch.bool),diagonal=1)
#torch.triu 会生成一个上三角矩阵（对角线以上为 True）。
print("未来位置掩码，True表示禁止关注：\n",future_mask)
#[False, True, True, ...] 表示：第 1 个词不能看第 2、3、4、5、6 个词

未来位置掩码，True表示禁止关注：
 tensor([[False,  True,  True,  True,  True,  True],
        [False, False,  True,  True,  True,  True],
        [False, False, False,  True,  True,  True],
        [False, False, False, False,  True,  True],
        [False, False, False, False, False,  True],
        [False, False, False, False, False, False]])


In [3]:
Q=torch.randn(1,seq_len,8); K=torch.randn(1,seq_len,8); V=torch.randn(1,seq_len,8)
scores=Q@K.transpose(-2,-1)/math.sqrt(Q.size(-1)); masked_scores=scores.masked_fill(future_mask,-1e9); weights=F.softmax(masked_scores,dim=-1); output=weights@V
print("注意力权重：\n",weights[0]); print("未来位置上的权重：\n",torch.triu(weights[0],diagonal=1)); print("输出形状：",output.shape)

注意力权重：
 tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2844, 0.7156, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1353, 0.3766, 0.4881, 0.0000, 0.0000, 0.0000],
        [0.0467, 0.1031, 0.7429, 0.1072, 0.0000, 0.0000],
        [0.3710, 0.2148, 0.0470, 0.2168, 0.1505, 0.0000],
        [0.2861, 0.5408, 0.0130, 0.0854, 0.0538, 0.0207]])
未来位置上的权重：
 tensor([[0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0.]])
输出形状： torch.Size([1, 6, 8])


## 2. 检查并加载本地 GPT

In [4]:
gpt_path=r"D:\11\NLP\data\gpt2-chinese-local"
gpt_ready=os.path.isdir(gpt_path) and os.path.isfile(os.path.join(gpt_path,"config.json")) and any(os.path.isfile(os.path.join(gpt_path,name)) for name in ["model.safetensors","pytorch_model.bin"])
print("GPT目录：",gpt_path); print("模型准备完成：",gpt_ready)
if os.path.isdir(gpt_path): print("目录文件：",os.listdir(gpt_path))

GPT目录： D:\11\NLP\data\gpt2-chinese-local
模型准备完成： True
目录文件： ['config.json', 'pytorch_model.bin', 'special_tokens_map.json', 'tokenizer_config.json', 'vocab.txt']


In [5]:
if gpt_ready:
    tokenizer=AutoTokenizer.from_pretrained(gpt_path,local_files_only=True)
    model=AutoModelForCausalLM.from_pretrained(gpt_path,local_files_only=True).to(device); model.eval()
    if tokenizer.pad_token_id is None: tokenizer.pad_token=tokenizer.eos_token
    print("词表大小：",len(tokenizer)); print("模型参数量：",sum(p.numel() for p in model.parameters()))
else: print("请先下载本地GPT模型。")

词表大小： 21128
模型参数量： 102068736


## 3. 查看输入 Token 和下一 Token 概率

In [6]:
prompt="人工智能的未来发展"
#给模型看一句话的前 N 个字，让它预测第 N+1 个字，然后与真实答案对比，错了就挨打（反向传播）
#GPT 没有“一次性写完”的能力，它只会“一个字一个字地往外蹦”。 这个过程叫自回归
if gpt_ready:
    inputs=tokenizer(prompt,return_tensors="pt").to(device); print("Tokens：",tokenizer.convert_ids_to_tokens(inputs["input_ids"][0]))
    with torch.no_grad(): logits=model(**inputs).logits
    next_probs=F.softmax(logits[0,-1],dim=-1); top_ids=torch.topk(next_probs,k=10).indices
    for token_id in top_ids: print(repr(tokenizer.decode([token_id.item()])),float(next_probs[token_id]))
else: print("模型未准备。")

Tokens： ['[CLS]', '人', '工', '智', '能', '的', '未', '来', '发', '展', '[SEP]']
'人' 0.09697074443101883
'（' 0.05528818443417549
':' 0.04650699719786644
'?' 0.03850255906581879
'？' 0.03818909078836441
'[SEP]' 0.02094276435673237
'(' 0.020145358517766
'：' 0.01869010366499424
'是' 0.01859443634748459
'，' 0.017922069877386093


四种常见的文本生成策略：贪婪搜索、Temperature + Top‑k、Temperature + Top‑p、束搜索（Beam Search）；

## 4. 贪婪搜索

In [7]:
if gpt_ready:
    with torch.no_grad(): greedy_ids=model.generate(**inputs,max_new_tokens=40,do_sample=False,pad_token_id=tokenizer.pad_token_id,eos_token_id=tokenizer.eos_token_id)
    print(tokenizer.decode(greedy_ids[0],skip_special_tokens=True))
else: print("模型未准备。")
'''贪婪搜索每一步都死盯着“当下概率最高”的那一个词。但它就像一个近视眼，只看脚下。'''

人 工 智 能 的 未 来 发 展 人 工 智 能 的 未 来 发 展


## 5. Temperature + Top-k

In [8]:
#Temperature（温度）+ Top-k 采样
if gpt_ready:
    torch.manual_seed(42)
    with torch.no_grad(): topk_ids=model.generate(**inputs,max_new_tokens=50,do_sample=True,temperature=0.8,top_k=30,pad_token_id=tokenizer.pad_token_id,eos_token_id=tokenizer.eos_token_id)
    #do_sample=True 意思是：不一定永远选择概率最高的那个词，而是按照概率随机抽取。
    #TemperatureGPT 生成文字时到底是“保守一点”还是“放飞一点”。
    '''Temperature 小,更保守,更倾向选择高概率词,输出更稳定
Temperature 大,更随机,低概率词也更容易出现,输出更有创造性'''
    '''top_k=30这是标题里的第二部分：Top-k
top_k=30意思是：每次生成下一个 Token 时，只考虑概率最高的 30 个 Token。'''
    print(tokenizer.decode(topk_ids[0],skip_special_tokens=True))
else: print("模型未准备。")

人 工 智 能 的 未 来 发 展 英 语 版idiv. 英 语 版 adlid.idiv / wise / wise / wise / wise / wise / wise / wise / wise /


## 6. Temperature + Top-p

In [9]:
#Top-p（核采样 / Nucleus Sampling）
#Top-p 的核心思想是：不是固定保留多少个候选词，而是从概率最高的词开始累加，直到累计概率达到 p。
#top_p=0.9就是：把概率从高到低排列，然后不断往下加，直到累计概率达到大约 90%，只留下这些 Token。  更灵活
if gpt_ready:
    torch.manual_seed(42)
    with torch.no_grad(): topp_ids=model.generate(**inputs,max_new_tokens=50,do_sample=True,temperature=0.8,top_p=0.9,pad_token_id=tokenizer.pad_token_id,eos_token_id=tokenizer.eos_token_id)
    print(tokenizer.decode(topp_ids[0],skip_special_tokens=True))
else: print("模型未准备。")

人 工 智 能 的 未 来 发 展 ( 长 按 图 片 识 别 二 维 码 ) [ photo ， url ， photo ， url ， url ， url ， url ， url ， url ， url ， url ， url ， url ， url ， url ， url ， url ， url ， url


## 7. Beam Search

In [10]:
#Beam Search（束搜索） 更像是在“同时保留几条候选句子，边生成边比较，最后选整体最好的那一条”
if gpt_ready:
    with torch.no_grad(): beam_ids=model.generate(**inputs,max_new_tokens=40,num_beams=4,early_stopping=True,no_repeat_ngram_size=2,pad_token_id=tokenizer.pad_token_id,eos_token_id=tokenizer.eos_token_id)
    '''num_beams=4意思是：同时维护 4 条最有希望的生成路线。num_beams=1基本就退化成：Greedy Search，贪心搜索。而：num_beams=4表示同时搜索 4 条路线。Beam数量越大,搜索越充分,更有可能找到高概率句子.但是Beam数量越大,计算越慢,显存/内存占用更多,不代表文本一定更自然'''
    print(tokenizer.decode(beam_ids[0],skip_special_tokens=True))
else: print("模型未准备。")

人 工 智 能 的 未 来 发 展 （ 人 机 交 互 ） 是 什 么 样 的 趋 势 ？ 这 是 一 个 很 有 趣 的 问 题 ， 不 知 道 有 没 有 人 能 回 答 一 下 ， 谢 谢


## 8. 对比不同 Temperature

In [11]:
if gpt_ready:
    for temperature in [0.3,0.7,1.2]:
        torch.manual_seed(42)
        with torch.no_grad(): ids=model.generate(**inputs,max_new_tokens=35,do_sample=True,temperature=temperature,top_p=0.9,pad_token_id=tokenizer.pad_token_id,eos_token_id=tokenizer.eos_token_id)
        print(f"temperature={temperature}：",tokenizer.decode(ids[0],skip_special_tokens=True))
else: print("模型未准备。")

temperature=0.3： 人 工 智 能 的 未 来 发 展 人 工 智 能 的 未 来 发 展 人 工 智 能 的 未 来 发 展
temperature=0.7： 人 工 智 能 的 未 来 发 展 ( ) 人 工 智 能 的 未 来 发 展 ( ) 人 工 智 能 的 未 来 发 展 ( ) 人 工 智 能 的 未 来 发 展 ( )
temperature=1.2： 人 工 智 能 的 未 来 发 展 英 语 版 发 布 新 概 念 语 言 与 技 术 引 擎 人 工 智 能 将 在 未 来 展 开 。 的 话 语 言 与 技 术 引


生成式预训练模型 = 一个提前在大量数据上学习过，并且能够根据输入继续生成内容的模型。

GPT = 使用 Transformer 架构、经过大规模预训练、能够生成内容的模型。

“预训练”就是先读大量资料学知识和语言规律；“生成式”就是学完以后，根据已有内容一个 Token 一个 Token 地继续往后写。